# <h1 align="center">Data Preprocessing - Data Cleaning</h1>


## PATIENT ID - HUPA0006P,HUPA0007P,HUPA0009P,HUPA0010P,HUPA0018P

In [2]:
# ============================================================
# Import Libraries
# ============================================================
import pandas as pd
import numpy as np

### 1.LOADED 5 FILES IN TO A SINGLE FILE FOR FURTHER CLEANING


In [1]:
import pandas as pd
import glob
import os

# Get CSV files from the folder
files_abi = glob.glob(
    r"C:\Users\vidya\OneDrive\Desktop\DATA ANALYST\NUMPY NINJA\GITHUB\Team2_-PyQueens_-Python-Hackathon-_MAY-2026\Abirami\MY_FILES_ABI\*.csv"
)

# Add patient_id column & merge all files
dfs = []

for file in files_abi:
    if os.path.isfile(file):
        df = pd.read_csv(file, sep=';')
        df['patient_id'] = os.path.basename(file).replace('.csv', '')
        dfs.append(df)

df_CGM5_ABI = pd.concat(dfs, ignore_index=True)

# Move patient_id to first column
cols = ['patient_id'] + [col for col in df_CGM5_ABI.columns if col != 'patient_id']
df_CGM5_ABI = df_CGM5_ABI[cols]

df_CGM5_ABI

,patient_id,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,HUPA0006P,2018-07-09T15:45:00,109.000000,11.23939,70.769231,0.0,0.075000,0.0,0.0
1,HUPA0006P,2018-07-09T15:50:00,107.666667,8.11090,70.189655,5.0,0.075000,0.0,0.0
2,HUPA0006P,2018-07-09T15:55:00,106.333333,6.48872,70.170940,0.0,0.075000,0.0,0.0
3,HUPA0006P,2018-07-09T16:00:00,105.000000,7.06807,72.368421,0.0,0.060833,0.0,0.0
4,HUPA0006P,2018-07-09T16:05:00,122.333333,9.61721,68.219298,6.0,0.060833,0.0,10.0
...,...,...,...,...,...,...,...,...,...
16825,HUPA0018P,2019-07-16T23:25:00,180.333333,6.56343,85.634328,0.0,0.000000,0.0,0.0
16826,HUPA0018P,2019-07-16T23:30:00,188.000000,4.94505,85.459259,0.0,0.000000,0.0,0.0
16827,HUPA0018P,2019-07-16T23:35:00,194.333333,5.12487,84.325581,0.0,0.000000,0.0,0.0
16828,HUPA0018P,2019-07-16T23:40:00,200.666667,9.35064,85.873950,0.0,0.000000,0.0,0.0


### 2 - CHECK: Duplicates

In [2]:
# ============================================================
#  2 — CHECK: Duplicates
# ============================================================
print("Full duplicate rows:", df_CGM5_ABI.duplicated().sum())
print("Duplicate timestamps per patient:")
print(df_CGM5_ABI.duplicated(subset=['patient_id', 'time']).sum())

Full duplicate rows: 0
Duplicate timestamps per patient:
0


#### Why: Duplicate rows mean the same reading is counted twice, causing inflated row counts and skewed statistics.
#### How it helps: Ensures every timestamp per patient is unique, giving accurate time-series analysis.
#### If skipped: Averages, totals, and trend lines would be distorted by repeated data points.

 ### 3-CHECK: Data Types

In [4]:
# ============================================================
# 3 — CHECK: Data Types
# ============================================================
print(df_CGM5_ABI.dtypes)

patient_id                 object
time                       object
glucose                   float64
calories                  float64
heart_rate                float64
steps                     float64
basal_rate                float64
bolus_volume_delivered    float64
carb_input                float64
dtype: object


In [5]:
# FIX: Data Types
df_CGM5_ABI['time'] = pd.to_datetime(df_CGM5_ABI['time'])

num_cols = ['glucose', 'calories', 'heart_rate', 'steps',
            'basal_rate', 'bolus_volume_delivered', 'carb_input']
df_CGM5_ABI[num_cols] = df_CGM5_ABI[num_cols].astype(float)

print("After fix:\n", df_CGM5_ABI.dtypes)

After fix:
 patient_id                        object
time                      datetime64[ns]
glucose                          float64
calories                         float64
heart_rate                       float64
steps                            float64
basal_rate                       float64
bolus_volume_delivered           float64
carb_input                       float64
dtype: object


#### Why: The 'time' column was loaded as plain text and numeric columns had inconsistent types, which prevents any time-based operations or mathematical calculations.
#### How it helps: Correct types allow sorting by time, calculating time differences, and performing accurate numeric aggregations.
#### If skipped: Operations like groupby time, plotting trends, or computing mean glucose would throw errors or produce wrong results.

### 4-CHECK: Missing / Null Values

In [6]:
# ============================================================
# 4 — CHECK: Missing / Null Values
# ============================================================
print("Null values per column:")
print(df_CGM5_ABI.isnull().sum())
print("\nTotal nulls:", df_CGM5_ABI.isnull().sum().sum())

# Check null % per column
print("\nNull percentage:")
print((df_CGM5_ABI.isnull().sum() / len(df_CGM5_ABI) * 100).round(2))

Null values per column:
patient_id                0
time                      0
glucose                   0
calories                  0
heart_rate                0
steps                     0
basal_rate                0
bolus_volume_delivered    0
carb_input                0
dtype: int64

Total nulls: 0

Null percentage:
patient_id                0.0
time                      0.0
glucose                   0.0
calories                  0.0
heart_rate                0.0
steps                     0.0
basal_rate                0.0
bolus_volume_delivered    0.0
carb_input                0.0
dtype: float64


#### Why: Null values in numeric columns cause errors in calculations and break visualisations.
#### How it helps: Confirming zero nulls gives confidence that the dataset is complete and no imputation is needed.
#### If skipped: Any null left in glucose or heart_rate would produce NaN results in all downstream calculations involving those columns.

### 5- CHECK: Trim & Clean Text

In [7]:
# ============================================================
# 5 — CHECK: Trim & Clean Text
# ============================================================
# Check whitespace in time column
print("Whitespace in time:")
print(df_CGM5_ABI['time'].astype(str).str.contains(r'^\s|\s$').sum())

# Check column name whitespace
print("\nColumn names:")
print([f"'{c}'" for c in df_CGM5_ABI.columns])

# Check patient_id formatting consistency
print("\nUnique patient IDs:")
print(df_CGM5_ABI['patient_id'].unique())

Whitespace in time:
0

Column names:
["'patient_id'", "'time'", "'glucose'", "'calories'", "'heart_rate'", "'steps'", "'basal_rate'", "'bolus_volume_delivered'", "'carb_input'"]

Unique patient IDs:
['HUPA0006P' 'HUPA0007P' 'HUPA0009P' 'HUPA0010P' 'HUPA0018P']


In [8]:
# ============================================================
# Check whitespace in any row values — df_CGM5_ABI
# ============================================================

print("df_CGM5_ABI — Whitespace check in all columns:")
for col in df_CGM5_ABI.columns:
    if df_CGM5_ABI[col].dtype == object:  # only text columns
        count = df_CGM5_ABI[col].astype(str).str.contains(r'^\s|\s$').sum()
        print(f"  {col}: {count} rows with whitespace")

df_CGM5_ABI — Whitespace check in all columns:
  patient_id: 0 rows with whitespace


#### Why: Leading or trailing whitespace in text columns like patient_id and time causes mismatches when filtering or grouping  'HUPA0006P ' and 'HUPA0006P' would be treated as two different patients.
#### How it helps: Clean text ensures accurate groupby operations, merges, and patient-level filtering.
#### If skipped: Patient groups would split incorrectly, producing wrong row counts and broken per-patient analysis.

### 6-CHECK: Standardize Values

In [9]:
# ============================================================
# 6- — CHECK: Standardize Values
# ============================================================
# Check sort order per patient
print("Is sorted by patient + time:")
sorted_check = df_CGM5_ABI.groupby('patient_id')['time'].is_monotonic_increasing
print(sorted_check)

# Check decimal precision
print("\nSample values:")
print(df_CGM5_ABI[num_cols].head(3))

Is sorted by patient + time:
patient_id
HUPA0006P    True
HUPA0007P    True
HUPA0009P    True
HUPA0010P    True
HUPA0018P    True
Name: time, dtype: bool

Sample values:
      glucose  calories  heart_rate  steps  basal_rate  \
0  109.000000  11.23939   70.769231    0.0       0.075   
1  107.666667   8.11090   70.189655    5.0       0.075   
2  106.333333   6.48872   70.170940    0.0       0.075   

   bolus_volume_delivered  carb_input  
0                     0.0         0.0  
1                     0.0         0.0  
2                     0.0         0.0  


In [12]:
# FIX: Round to 2 decimal places
num_cols = ['glucose', 'calories', 'heart_rate', 'steps',
            'basal_rate', 'bolus_volume_delivered', 'carb_input']

df_CGM5_ABI[num_cols] = df_CGM5_ABI[num_cols].round(2)

print("After rounding:")
print(df_CGM5_ABI[num_cols].head(3))
df_CGM5_ABI.head(3)

After rounding:
   glucose  calories  heart_rate  steps  basal_rate  bolus_volume_delivered  \
0   109.00     11.24       70.77    0.0        0.08                     0.0   
1   107.67      8.11       70.19    5.0        0.08                     0.0   
2   106.33      6.49       70.17    0.0        0.08                     0.0   

   carb_input  
0         0.0  
1         0.0  
2         0.0  


,patient_id,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,HUPA0006P,2018-07-09 15:45:00,109.00,11.24,70.77,0.0,0.08,0.0,0.0
1,HUPA0006P,2018-07-09 15:50:00,107.67,8.11,70.19,5.0,0.08,0.0,0.0
2,HUPA0006P,2018-07-09 15:55:00,106.33,6.49,70.17,0.0,0.08,0.0,0.0


#### Why: Data was not guaranteed to be in chronological order after merging, and numeric columns had up to 9 decimal places making the data inconsistent and hard to read.
#### How it helps: Sorting ensures time-series operations like diff() and interpolation work correctly. Rounding to 2 decimal places standardises precision across all patients.
#### If skipped: Time-based calculations like glucose trend detection or interval checks would produce incorrect results on unsorted data. Excessive decimals would cause inconsistency when comparing values across patients.

### 7-CHECK: Column Names

In [18]:
# ============================================================
# 7 — CHECK: Column Names
# ============================================================
print("Current column names:")
print(df_CGM5_ABI.columns.tolist())

Current column names:
['patient_id', 'Time', 'Glucose', 'Calories', 'Heart_Rate', 'Steps', 'Basal_Rate', 'Bolus_Volume_Delivered', 'Carb_Input']


In [13]:
# FIX: Rename Columns
df_CGM5_ABI = df_CGM5_ABI.rename(columns={
    'time'                   : 'Time',
    'glucose'                : 'Glucose',
    'calories'               : 'Calories',
    'heart_rate'             : 'Heart_Rate',
    'steps'                  : 'Steps',
    'basal_rate'             : 'Basal_Rate',
    'bolus_volume_delivered' : 'Bolus_Volume_Delivered',
    'carb_input'             : 'Carb_Input'
    
})
print("Renamed:", df_CGM5_ABI.columns.tolist())

Renamed: ['patient_id', 'Time', 'Glucose', 'Calories', 'Heart_Rate', 'Steps', 'Basal_Rate', 'Bolus_Volume_Delivered', 'Carb_Input']


#### Why: Original column names like 'bolus_volume_delivered' were lowercase with underscores and lacked units, making them harder to read in reports and dashboards.
#### How it helps: Descriptive, consistently capitalised column names improve readability in Power BI, charts, and presentation outputs.
#### If skipped: Dashboards and reports would display raw technical names that are unclear to non-technical stakeholders.

### 8-Inconsistent Categories

In [14]:
# ============================================================
# 8-Inconsistent Categories
# ============================================================
# Validate zero distributions per patient
print("Zero % per column per patient:")
zero_pct = (
    df_CGM5_ABI.groupby('patient_id')[df_CGM5_ABI.columns[2:]]
    .apply(lambda x: (x == 0).sum() / len(x) * 100)
    .round(1)
)
print(zero_pct)

Zero % per column per patient:
            Glucose  Calories  Heart_Rate  Steps  Basal_Rate  \
patient_id                                                     
HUPA0006P       0.0       0.0         0.0   67.8         1.8   
HUPA0007P       0.0       0.0         0.0   59.0         6.6   
HUPA0009P       0.0       0.0         0.0   58.8         1.4   
HUPA0010P       0.0       0.0         0.0   53.4         3.3   
HUPA0018P       0.0       0.0         0.0   61.3       100.0   

            Bolus_Volume_Delivered  Carb_Input  
patient_id                                      
HUPA0006P                     98.0        98.4  
HUPA0007P                     97.7        97.9  
HUPA0009P                     98.3        98.8  
HUPA0010P                     97.4        98.6  
HUPA0018P                    100.0       100.0  


#### Why: In datasets with categorical columns, inconsistent formats like 'Male', 'male', 'M' would be treated as three separate categories, breaking group analysis.
#### How it helps: Confirming no categorical columns exist and that zeros are clinically valid prevents unnecessary data modification.
#### If skipped: Incorrectly treating valid zeros as missing data and replacing them would corrupt clinically meaningful readings like no bolus delivered or no food logged.

### 9 -  CHECK: Validate Relationships / Keys

In [78]:
# ============================================================
# 9- CHECK: Validate Relationships / Keys
# ============================================================
# Timestamp uniqueness per patient
dup_check = df_CGM5_ABI.duplicated(subset=['patient_id', 'Time']).sum()
print("Duplicate patient+timestamp combinations:", dup_check)

# Row count per patient
print("\nRows per patient:")
print(df_CGM5_ABI['patient_id'].value_counts())

# Time interval check per patient
print("\nTime interval check (should all be 5 min):")
df_CGM5_ABI_sorted = df_CGM5_ABI.sort_values(['patient_id', 'Time'])
intervals = (
    df_CGM5_ABI_sorted.groupby('patient_id')['Time']
    .diff()
    .dropna()
    .value_counts()
)
print(intervals.head(10))

Duplicate patient+timestamp combinations: 0

Rows per patient:
patient_id
HUPA0018P    3895
HUPA0007P    3857
HUPA0009P    3812
HUPA0010P    2976
Name: count, dtype: int64

Time interval check (should all be 5 min):
Time
0 days 00:05:00    14536
Name: count, dtype: int64


#### Why: As a time-series dataset, each patient must have unique timestamps and consistent 5-minute intervals. Any break in this structure affects trend analysis and machine learning models.
#### How it helps: Confirms the dataset is structurally sound  no duplicate keys, no missing intervals, and correct row counts per patient after merging.
#### If skipped: Duplicate timestamps would cause incorrect joins and aggregations. Irregular time intervals would break any model or visualisation that assumes evenly spaced readings.

### 10 — Final Summary 

In [15]:
# ============================================================
# 10 — Final Summary
# ============================================================
print("Final Shape:", df_CGM5_ABI.shape)
print("\nData Types:\n", df_CGM5_ABI.dtypes)
print("\nNull Values:\n", df_CGM5_ABI.isnull().sum())
print("\nPatients:", df_CGM5_ABI['patient_id'].unique())
df_CGM5_ABI.describe()

Final Shape: (16830, 9)

Data Types:
 patient_id                        object
Time                      datetime64[ns]
Glucose                          float64
Calories                         float64
Heart_Rate                       float64
Steps                            float64
Basal_Rate                       float64
Bolus_Volume_Delivered           float64
Carb_Input                       float64
dtype: object

Null Values:
 patient_id                0
Time                      0
Glucose                   0
Calories                  0
Heart_Rate                0
Steps                     0
Basal_Rate                0
Bolus_Volume_Delivered    0
Carb_Input                0
dtype: int64

Patients: ['HUPA0006P' 'HUPA0007P' 'HUPA0009P' 'HUPA0010P' 'HUPA0018P']


,Time,Glucose,Calories,Heart_Rate,Steps,Basal_Rate,Bolus_Volume_Delivered,Carb_Input
count,16830,16830.000000,16830.000000,16830.000000,16830.000000,16830.000000,16830.000000,16830.000000
mean,2018-11-29 22:16:18.110516992,163.323878,9.380301,76.121116,41.813428,0.060240,0.054743,0.038963
min,2018-07-09 15:45:00,40.000000,4.070000,41.420000,0.000000,0.000000,0.000000,0.000000
25%,2018-09-22 22:21:15,107.330000,4.680000,65.190000,0.000000,0.000000,0.000000,0.000000
50%,2018-09-30 05:40:00,158.330000,6.930000,76.745000,0.000000,0.070000,0.000000,0.000000
75%,2018-11-16 21:28:45,214.000000,11.070000,84.477500,43.000000,0.090000,0.000000,0.000000
max,2019-07-16 23:45:00,438.000000,78.860000,166.330000,682.000000,0.150000,13.300000,10.000000
std,NaN,71.129606,7.285117,14.450585,86.712816,0.039456,0.567493,0.435361


### 11 — Moving the patient id as first column 

In [ ]:
# ============================================================
# 11 — Moving the patient id as first column
# ============================================================

In [17]:
# Move patient_id to first column
cols = ['patient_id'] + [col for col in df_CGM5_ABI.columns if col != 'patient_id']
df_CGM5_ABI = df_CGM5_ABI[cols]

print(df_CGM5_ABI.columns.tolist())
df_CGM5_ABI.head()

['patient_id', 'Time', 'Glucose', 'Calories', 'Heart_Rate', 'Steps', 'Basal_Rate', 'Bolus_Volume_Delivered', 'Carb_Input']


,patient_id,Time,Glucose,Calories,Heart_Rate,Steps,Basal_Rate,Bolus_Volume_Delivered,Carb_Input
0,HUPA0006P,2018-07-09 15:45:00,109.00,11.24,70.77,0.0,0.08,0.0,0.0
1,HUPA0006P,2018-07-09 15:50:00,107.67,8.11,70.19,5.0,0.08,0.0,0.0
2,HUPA0006P,2018-07-09 15:55:00,106.33,6.49,70.17,0.0,0.08,0.0,0.0
3,HUPA0006P,2018-07-09 16:00:00,105.00,7.07,72.37,0.0,0.06,0.0,0.0
4,HUPA0006P,2018-07-09 16:05:00,122.33,9.62,68.22,6.0,0.06,0.0,10.0


###  12 — Save Cleaned File to CSV File

In [ ]:
# ============================================================
# 12 — Save Cleaned File
# ============================================================
df_CGM5_ABI.to_csv(
    r"C:\Users\vidya\OneDrive\Desktop\DATA ANALYST\NUMPY NINJA\GITHUB\Team2_-PyQueens_-Python-Hackathon-_MAY-2026\Abirami\MY_FILES_ABI\CGM5_ABI_cleaned.csv",
    index=False
)
print("Saved successfully!")